In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import Patch

from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.callbacks import EvalCallback

from src.data_processor import add_technical_indicators, extract_regimes
from src.environment import OptimalExecutionEnv

In [ ]:
df_raw = pd.read_csv("data/BTCUSDT_1h_FINAL.csv")
df_raw['Open_Time'] = pd.to_datetime(df_raw['Open_Time'])
print(f"Loaded: {len(df_raw):,} rows | {df_raw['Open_Time'].min()} → {df_raw['Open_Time'].max()}")

df_enriched = add_technical_indicators(df_raw)
df_final, transmat = extract_regimes(df_enriched)
print("\nTransition matrix:")
print(np.round(transmat, 4))

In [ ]:
split_idx = int(len(df_final) * 0.8)
train_df  = df_final.iloc[:split_idx].reset_index(drop=True)
test_df   = df_final.iloc[split_idx:].reset_index(drop=True)

print(f"Train: {len(train_df):,} rows  ({train_df['Open_Time'].min()} → {train_df['Open_Time'].max()})")
print(f"Test:  {len(test_df):,} rows  ({test_df['Open_Time'].min()} → {test_df['Open_Time'].max()})")

In [ ]:
T = 24
TOTAL_TO_SELL = 10.0

def make_env(df, transmat, T, total_to_sell):
    def _init():
        env = OptimalExecutionEnv(df, transmat, total_to_sell=total_to_sell, T=T)
        return Monitor(env)
    return _init


train_env = DummyVecEnv([make_env(train_df, transmat, T, TOTAL_TO_SELL)])
train_env = VecNormalize(train_env, norm_obs=True, norm_reward=True, clip_obs=10.0)

eval_vec = DummyVecEnv([make_env(test_df, transmat, T, TOTAL_TO_SELL)])
eval_vec = VecNormalize(eval_vec, norm_obs=True, norm_reward=False, training=False, clip_obs=10.0)

model = PPO(
    "MlpPolicy", train_env,
    learning_rate=3e-4, n_steps=2048, batch_size=256, n_epochs=10,
    gamma=0.99, gae_lambda=0.95, clip_range=0.2, ent_coef=0.01,
    policy_kwargs=dict(net_arch=[256, 256, 128]),
    verbose=1,
)

eval_cb = EvalCallback(eval_vec, best_model_save_path="./models/",
                       eval_freq=10_000, n_eval_episodes=5,
                       deterministic=True, verbose=0)

print("Entraînement en cours…")

model.learn(total_timesteps=200_000, callback=eval_cb, progress_bar=True)

In [ ]:
def run_episode(strategy, env, model=None, vec_norm=None):
    obs, _ = env.reset()
    done, step = False, 0
    history = {'price': [], 'amount': [], 'inventory': [], 'regime': [], 'revenue': []}
    while not done:
        if strategy == 'drl':
            obs_n = vec_norm.normalize_obs(obs.reshape(1, -1))
            action, _ = model.predict(obs_n, deterministic=True)
        elif strategy == 'twap':
            frac   = 1.0 / env.T
            action = np.array([np.clip(frac / max(env.inventory / env.initial_inventory, 1e-6), 0, 1)])
        elif strategy == 'vwap':
            action = np.array([1.0 / max(env.T - step, 1)])
        obs, reward, done, _, info = env.step(action)
        history['price'].append(info['price'])
        history['amount'].append(info['amount'])
        history['inventory'].append(info['inventory'])
        history['regime'].append(int(round(float(obs[5]))))
        history['revenue'].append(info['revenue'])
        step += 1
    history['total_revenue'] = sum(history['revenue'])
    return history


def test_agent(model, vec_norm, test_df, transmat, n_episodes=5):
    REGIME_COLORS = {0: '#e74c3c', 1: '#f39c12', 2: '#2ecc71'}
    REGIME_LABELS = {0: 'Bearish', 1: 'Neutral', 2: 'Bullish'}
    strategies    = ['drl', 'twap', 'vwap']
    all_revenues  = {s: [] for s in strategies}

    for ep in range(n_episodes):
        seed = ep * 7
        envs = {s: OptimalExecutionEnv(test_df, transmat, total_to_sell=10.0, T=24)
                for s in strategies}
        envs['drl'].reset(seed=seed)
        tick = envs['drl'].start_tick
        for s, env in envs.items():
            env.start_tick = tick; env.current_step = 0; env.inventory = env.initial_inventory

        for s in strategies:
            h = run_episode(s, envs[s], model=model, vec_norm=vec_norm)
            all_revenues[s].append(h['total_revenue'])

        print(f"Ép.{ep+1}: DRL={all_revenues['drl'][-1]:>12,.0f}  "
              f"TWAP={all_revenues['twap'][-1]:>12,.0f}  "
              f"VWAP={all_revenues['vwap'][-1]:>12,.0f} USDT")
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    colors = ['#3498db', '#e67e22', '#9b59b6']
    data = [all_revenues[s] for s in strategies]
    bp   = axes[0].boxplot(data, patch_artist=True, widths=0.4,
                           medianprops=dict(color='white', linewidth=2))
    for patch, c in zip(bp['boxes'], colors):
        patch.set_facecolor(c); patch.set_alpha(0.8)
    axes[0].set_xticklabels(['DRL', 'TWAP', 'VWAP'])
    axes[0].set_title("Distribution des revenus", fontweight='bold')
    axes[0].set_ylabel("Revenu Total (USDT)")
    axes[0].spines[['top', 'right']].set_visible(False)
    x = range(1, n_episodes + 1)
    for s, c in zip(strategies, colors):
        axes[1].plot(x, all_revenues[s], marker='o', label=s.upper(), color=c, lw=2)
    axes[1].set_title("Revenu par épisode (USDT réels)", fontweight='bold')
    axes[1].set_xlabel("Épisode"); axes[1].set_ylabel("USDT")
    axes[1].legend(); axes[1].spines[['top', 'right']].set_visible(False)

    plt.suptitle("Performance de l'Agent — DRL vs TWAP vs VWAP", fontweight='bold')
    plt.tight_layout()
    plt.show()

In [ ]:
print("\nÉvaluation de l'agent")
test_agent(model, eval_vec, test_df, transmat, n_episodes=5)